# သင်ခန်းစာ 05 - Agentic RAG


## Setup

ဤနိုတ်ဘုတ်တွင် Microsoft Agent Framework ကို အသုံးပြု၍ Agentic RAG (Retrieval-Augmented Generation) ပုံစံကို ဖော်ပြထားသည်။

**လိုအပ်သောအချက်များ**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — သင့် Azure AI Search ဝန်ဆောင်မှု endpoint
- `AZURE_SEARCH_API_KEY` — သင့် Azure AI Search API key
- environment variables ဖြင့် Azure OpenAI deployment ကို တပ်ဆင်ထားရန်
- Azure CLI ဖြင့် authentication ပြုလုပ်ထားရန် (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG ဆိုတာဘာလဲ?

ရိုးရာ RAG သည် အတည်ပြုထားသော ပိုင်းလိုင်းတစ်ခုကို လိုက်နာသည်။ အရင်ဆုံး စာရွက်စာတမ်းများကို ရွေးချယ်ပြီး ထို့နောက် ပြန်တမ်းတစ်ခု ထုတ်ပေးသည်။ **Agentic RAG** သည် အေးဂျင့်အား သတင်းအချက်အလက် ရယူရန် **ဘယ်အချိန်**နှင့် **ဘယ်လို** ဆောင်ရွက်မည်ကို ကိုယ်တိုင်ဆုံးဖြတ်ခွင့်ပေးခြင်းဖြင့် ပိုမိုတိုးတက်စေသည်။

Agentic RAG ဖြင့် အေးဂျင့်သည် -
- မေးခွန်းကိုဖြေရှင်းရန်မတိုင်မီ ရယူမှု လိုအပ်မလားကို **ဆုံးဖြတ်** ပေးနိုင်သည်
- ရှာဖွေရန် ဒေတာအရင်းအမြစ် သို့မဟုတ် ကိရိယာကို **ရွေးချယ်** နိုင်သည်
- ရရှိသော ရလဒ်များကို **သုံးသပ်** လုပ်၍ ပထမဆုံးကြိုးစားချက် မလုံလောက်ပါက အောက်မေ့ ရယူမှုများ ဆက်လက်ဆောင်ရွက်နိုင်သည်
- ရယူမှု အဆင့်အတန်းများစွာမှ သတင်းအချက်အလက်များကို **ပေါင်းစပ်** သာယာ၍ဖြေကြားချက်တစ်ခုအဖြစ်ဖော်ထုတ်နိုင်သည်

ဤကဲ့သို့ဖြင့် အေးဂျင့် သည် ရှေ့ပြေး retrieve-then-generate ပိုင်းလိုင်းထက် ပိုမိုချောမွေ့ပြီး တိကျမှန်ကန်မှုများရှိစေသည်။


## ရှာဖွေရန်ကိရိယာတစ်ခု ဖန်တီးခြင်း

Agentic RAG တွင် ပြင်ပဒေတာအရင်းအမြစ်များကို ကိုယ်စားလှယ်သည် တောင်းဆိုသူ အလိုအလျောက် ခေါ်ယူနိုင်သော **ကိရိယာများ** အဖြစ် သိုလှောင်ထားသည်။ ဤကိရိယာများအား အသုံးပြုခြင်းဖြင့် ကိုယ်စားလှယ်သည် ရယူခြင်းကို လုပ်ဆောင်နိုင်သော အခြားလှုပ်ရှားမှုတစ်ခုအဖြစ်ယူဆနိုင်ပြီး အလိုအလျောက်လိုအပ်သော အဆင့်တစ်ခု မဟုတ်တော့ပါ။

အောက်တွင် ခရီးသွားအသိပညာ ဘတ်(စ်)ကို သတ်မှတ်ပြီး ကိုယ်စားလှယ်သည် ခရီးသွားဂရပ်ဖ်သတင်းအချက်အလက်ကို ရှာဖွေခေါ်ယူနိုင်သော ကိရိယာအဖြစ် ဖော်ပြထားပါသည်။


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG အေးဂျင့်တည်ဆောက်ခြင်း

ယခုမှာ အဖြေပြန်ရန်မပြုမီ **အမြဲ အချက်အလက်ရှာဖွေရန်** ညွှန်ကြားထားသော အေးဂျင့်တစ်ခု ဖန်တီးသည်။ အဲ့ဒီအေးဂျင့်သည် `search_travel_knowledge` ကိရိယာကို အသုံးပြုကာ ကိုယ်ပိုင်သင်တန်းဒေတာအစား အသိပညာအခြေခံမှတဆင့် သူ၏တုံ့ပြန်ချက်များကို အခြေခံသည်။


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## 반복적인 검색 — 제작자-검수자 패턴

Agentic RAG의 주요 장점은 **반복적인 검색**입니다. 에이전트는 초기 발견 내용을 확인, 수정 또는 확장하기 위해 여러 번 검색 작업을 수행할 수 있습니다 — "제작자-검수자" 작업 흐름과 유사합니다:

1. **제작자 단계**: 에이전트가 초기 정보를 검색하고 응답 초안을 작성합니다.
2. **검수자 단계**: 에이전트가 세부 사항을 검증하거나 누락된 부분을 채우기 위해 추가 검색을 수행합니다.

아래 예시에서 에이전트는 여러 목적지를 비교해야 하는 질문을 받아 여러 번 검색하도록 유도됩니다.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## အကျဉ်းချုပ်

ဒီသင်ခန်းစာမှာ Microsoft Agent Framework ကို အသုံးပြုပြီး **Agentic RAG** စနစ်ကို ဘယ်လိုတည်ဆောက်ရမလဲ သိရှိခဲ့သည်။

- **Agentic RAG** သည်အေးဂျင့်များအား သတင်းအချက်အလက်ရှာဖွေမှုကို မျှော်မှန်းထားသည့်အချိန်အစား၊ ကိုယ်ပိုင်ဆုံးဖြတ်ချက်ဖြင့် ထုတ်ယူနိုင်စေသည်။
- **ကိရိယာများကို ဒေတာအရင်းအမြစ်အဖြစ် အသုံးပြုခြင်း**။ ပြင်ပအသိပညာအခြေခံ (Azure AI Search ကဲ့သို့) ကို အေးဂျင့်က အသုံးပြုနိုင်မယ့် ကိရိယာအဖြစ် ထည့်သွင်းထားသည်။
- **စဉ်ဆက်မပြတ်ထုတ်ယူခြင်း**။ maker-checker ပုံစံက အေးဂျင့်အား ရှာဖွေခြင်း၊ အတည်ပြုခြင်းနှင့် တိုးတက်ပြုပြင်ခြင်းတို့ကို အကြိမ်အနည်းငယ်ပြုလုပ်ပြီး အဆုံးသတ်ဖြေကြားချက်ထုတ်ပေးနိုင်စေသည်။

ထုတ်လုပ်မှုတွင် သင်သည် in-memory `TRAVEL_KNOWLEDGE_BASE` ကို အစားလွှဲ၍ ကြီးမားသော ခရီးသွားစာရွက်စာတမ်းရှာဖွေရန် အတွက် အမှန်တကယ် Azure AI Search အညွှန်းထားကို အသုံးပြုမည်ဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**အတည်မပြုချက်**  
ဤစာတမ်းကို AI ဘာသာပြန်ဆောင်းမှုဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ တိကျမှန်ကန်မှုအတွက် ကြိုးစားပြီးဖြစ်သော်လည်း၊ အလိုအလျောက်ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် မှန်ကန်မှုမရှိမှုများ ရှိနိုင်ကြောင်း သတိပြုပါ။ မူလစာတမ်းသည် မိခင်ဘာသာဖြင့် ရှိသည်မှာ တရားဝင်အချက်အလက်ဖြစ်ပါသည်။ အရေးကြီးသော သတင်းအချက်အလက်များအတွက် အတတ်ပညာအရ ဘာသာပြန်သူလူကိုယ်တိုင်၏ ဘာသာပြန်ချက်ကို အသုံးပြုရန်အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာနိုင်သည့် နားမလည်မှု သို့မဟုတ် မှားယွင်းဖော်ပြချက်များအတွက် ကျွန်ုပ်တို့သည် တာဝန်မကျေဘူးဖြစ်ပါသည်။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
